# Lecture 15 - Stacks, queues and recursion

**Lecture Topic**: Two classic structures with opposite rules, the call stack behind every error you will ever read, and recursion.

## Announcements

- Module C continues.
- Keep going on your project work.
- ClassBuzz poll for today: `Q8FZ`

## Recap: last time

- **Array**: contiguous, O(1) to index, costly to grow at the front.
- **Linked list**: nodes with references, O(1) at the front, O(n) to index.
- **Hash table** (dict, set): average O(1) membership, and what pandas indexes and joins run on.

Today: two structures built on those, **stacks** and **queues**, the **call stack** that is behind your code, and **recursion**.

Today:
 - Stacks (last in, first out)
 - Queues (first in, first out), and why a list is a bad queue
 - The call stack and reading tracebacks
 - Recursion, its depth limit, and the iterative rewrite

# Stacks: last in, first out

A **stack** is a linear structure with a single access point, the **top**. You can `push` an item onto the top, `pop` the top item off, and `peek` at it without removing it. The last thing in is the first thing out (LIFO), like a stack of plates.

A Python list already is a stack: `append` is push, `pop()` is pop, `[-1]` is peek, and all three are O(1).

### Question 1

A program pushes these values onto a stack in this order: `10, 20, 30, 0, -30`. You then call `pop()` five times. What sequence comes out?

 - A. `10, 20, 30, 0, -30`
 - B. `-30, 0, 10, 20, 30`
 - C. `30, 10, 20, 0, -30`
 - D. `-30, 0, 30, 20, 10`

In [ ]:
stack = []
for v in [10, 20, 30, 0, -30]:
    stack.append(v)        # push

out = [stack.pop() for _ in range(5)]   # pop five times
print(out)

**D, `-30, 0, 30, 20, 10`.** A stack reverses order: the last item pushed (`-30`) is the first popped. That reversal is exactly what makes a stack useful.

## What stacks are for

The classic use is **undo**: every edit is pushed onto a stack, and undo pops the most recent one. Another is checking that brackets are balanced, which is the same shape as how an editor or a parser validates code.

In [ ]:
def brackets_balanced(s):
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        if ch in "([{":
            stack.append(ch)               # push an opener
        elif ch in ")]}":
            if not stack or stack.pop() != pairs[ch]:
                return False               # closer with no matching opener
    return not stack                        # leftover openers = unbalanced

for s in ["(a[0] + b[1])", "([)]", "df.loc[x, (a, b)]", "("]:
    print(f"{s!r:22s} -> {brackets_balanced(s)}")

The stack holds the openers you have seen but not yet closed. A closer has to match the most recent opener, which is the one on top. If anything is left on the stack at the end, some opener was never closed.

# Queues: first in, first out

A **queue** has two access points: you `enqueue` at the rear and `dequeue` from the front. The first thing in is the first thing out (FIFO), like a line of people. Autograders, print spoolers, and task schedulers are queues.

| | Stack | Queue |
|---|---|---|
| Order | LIFO | FIFO |
| Access points | one (top) | two (front, rear) |
| Add / remove | push / pop | enqueue / dequeue |

## Why a Python list makes a bad queue

You might implement a queue with a list: `append` to enqueue, `pop(0)` to dequeue from the front. It works, but `pop(0)` has to shift every remaining element down one slot, so it is O(n), the same front-of-array cost we saw last time. Over `n` items the queue is quietly O(n^2).

`collections.deque` is built for this: O(1) at **both** ends.

In [ ]:
import time
from collections import deque

def drain_list(n):
    q = list(range(n))
    while q:
        q.pop(0)             # O(n) each time: shifts everything down

def drain_deque(n):
    q = deque(range(n))
    while q:
        q.popleft()          # O(1) each time

for f in [drain_list, drain_deque]:
    start = time.perf_counter()
    f(50_000)
    print(f"{f.__name__:12s}: {time.perf_counter() - start:.3f} s")

### Question 2

You implement a queue with a Python list, using `append` to add and `pop(0)` to remove. Why does it scale badly?

 - A. `append` is O(n)
 - B. `pop(0)` is O(n), because every other element shifts down
 - C. Lists cannot hold more than a few thousand items
 - D. It does not scale badly; it is fine

**B.** Removing from the front of a contiguous list shifts all the remaining elements, so each `pop(0)` is O(n) and draining the queue is O(n^2). Reach for `collections.deque` whenever you need a queue, or anytime you are adding and removing at both ends.

# The call stack: the stack you are already using

Every running program keeps a stack of its own. Each time a function is called, a **frame** is pushed onto the **call stack** holding that call's local variables and the place to return to. When the function returns, its frame is popped.

You have been reading this stack all term, because a **traceback** is a printout of the call stack at the moment something went wrong.

In [ ]:
import traceback

def load(row):
    return parse(row)

def parse(row):
    return 100 / row["count"]        # row["count"] is 0

try:
    load({"count": 0})
except ZeroDivisionError:
    print(traceback.format_exc())

Read it from the **bottom up**: the last line is the actual error, and the frames above it are the chain of calls that led there (`load` called `parse`, `parse` did the division). The bottom is almost always the most useful line. This is the single most valuable debugging habit, and it is just reading a stack.

# Recursion

A **recursive** function calls itself on a smaller version of the problem until it reaches a **base case** that it can answer directly. Each call pushes a new frame onto the call stack.

In [ ]:
def factorial(n):
    if n <= 1:          # base case
        return 1
    return n * factorial(n - 1)      # recursive case: a smaller problem

print([factorial(k) for k in range(6)])

## The depth limit

Because every call adds a frame, deep recursion can exhaust the call stack. Python guards against that with a recursion limit (about 1000 frames by default) and raises `RecursionError` rather than crashing.

In [ ]:
import sys
print("recursion limit:", sys.getrecursionlimit())

def countdown(n):
    if n == 0:
        return 0
    return countdown(n - 1)

try:
    countdown(100_000)               # far deeper than the limit
except RecursionError as e:
    print("RecursionError:", e)

### Question 3

A recursive function is called with an input far deeper than the recursion limit, but every call is correct and does reach a base case eventually. What happens?

 - A. It returns the right answer, just slowly
 - B. It raises `RecursionError` before finishing
 - C. It silently returns `None`
 - D. Python automatically converts it to a loop

**B.** Correctness is not the issue; depth is. Each pending call holds a frame, and once there are more frames than the limit, Python raises `RecursionError`. Python does not optimise tail calls into loops, so depth really is bounded.

## The iterative rewrite

Anything recursive can be written as a loop, and the loop uses O(1) stack space instead of one frame per step. When recursion depth would grow with the data (walking a long chain, a deep file tree), the loop is the safe choice.

In [ ]:
def factorial_iter(n):
    result = 1
    for k in range(2, n + 1):
        result *= k
    return result

print("recursive == iterative:",
      all(factorial_iter(k) == factorial(k) for k in range(1, 20)))
print("iterative handles big n fine:", factorial_iter(5000) > 0)

Recursion is the natural way to express anything tree-shaped, and we will lean on it next time for binary search trees, where the depth is only about `log n` and the limit is no danger. The rule of thumb: recursion for shallow, tree-shaped problems; a loop when the depth could grow with the size of the data.

# Where this meets real data: flood fill

Here is a task that is naturally recursive: given a grid, measure the size of a connected region, meaning the seed cell, its neighbours, their neighbours, and so on. This is **flood fill**, the same idea as a paint-bucket tool, and it is how you label connected areas in an image or a gridded dataset.

We will run it on real data: the cells of the climate grid that warmed between 1851 and 2100, which turn out to form one large connected blob.

In [ ]:
import numpy as np
import xarray as xr

ds = xr.open_dataset("../data/global_temparature.nc")
warm = ds["tas"].isel(time=-1).values > ds["tas"].isel(time=0).values   # 96 x 144 of True/False
print("warm cells:", int(warm.sum()), "of", warm.size)

seed = tuple(np.argwhere(warm)[0])      # a cell inside the warm region

def region_size_recursive(grid, seen, r, c):
    if r < 0 or r >= grid.shape[0] or c < 0 or c >= grid.shape[1]:
        return 0
    if seen[r, c] or not grid[r, c]:
        return 0
    seen[r, c] = True
    return (1
            + region_size_recursive(grid, seen, r + 1, c)
            + region_size_recursive(grid, seen, r - 1, c)
            + region_size_recursive(grid, seen, r, c + 1)
            + region_size_recursive(grid, seen, r, c - 1))

try:
    size = region_size_recursive(warm, np.zeros_like(warm), *seed)
    print("recursive region size:", size)
except RecursionError:
    print("RecursionError: the region is deeper than Python's recursion limit allows")

The recursion runs straight into the limit from earlier in the lecture, because filling one cell calls itself on its neighbours, and the chain of pending calls is thousands deep on a region this big. The logic is correct; the depth is the problem.

## The same work with your own stack

Any recursion is a loop plus a stack you keep yourself. Hold a list of cells still to visit, pop one, mark it, and push its neighbours. Popping from the end makes it depth-first, exactly like the recursion, except the depth now lives in a list on the heap, so there is no limit to hit.

In [ ]:
def region_size_iterative(grid, seed):
    seen = np.zeros_like(grid)
    stack = [seed]                      # our own stack of cells still to visit
    size = 0
    while stack:
        r, c = stack.pop()              # pop from the end: depth-first
        if r < 0 or r >= grid.shape[0] or c < 0 or c >= grid.shape[1]:
            continue
        if seen[r, c] or not grid[r, c]:
            continue
        seen[r, c] = True
        size += 1
        stack.extend([(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)])
    return size

print("iterative (stack) region size:", region_size_iterative(warm, seed))

In [ ]:
from collections import deque
from scipy import ndimage

def region_size_bfs(grid, seed):
    seen = np.zeros_like(grid)
    queue = deque([seed])               # a queue instead of a stack
    size = 0
    while queue:
        r, c = queue.popleft()          # popleft: breadth-first
        if not (0 <= r < grid.shape[0] and 0 <= c < grid.shape[1]):
            continue
        if seen[r, c] or not grid[r, c]:
            continue
        seen[r, c] = True
        size += 1
        queue.extend([(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)])
    return size

labels, n_regions = ndimage.label(warm)                 # the library version
seed_region = int((labels == labels[seed]).sum())
print("iterative (deque) region size:", region_size_bfs(warm, seed))
print("scipy.ndimage.label:", n_regions, "region(s); the seed's region has", seed_region, "cells")

The stack version and the `deque` version agree, and both match `scipy.ndimage.label`, which is the function you should actually use!

# Recap

 - A **stack** is LIFO (push/pop/peek, all O(1)); a Python list is one. Good for undo and matching.
 - A **queue** is FIFO; a list is a *bad* queue because `pop(0)` is O(n). Use `collections.deque`.
 - The **call stack** holds a frame per active call; a **traceback** is that stack, and you read it bottom-up.
 - **Recursion** pushes a frame per call, so it has a depth limit; rewrite as a loop when depth grows with the data.
 - **Flood fill** on the climate grid shows all of this at once: the recursive version hits the limit, and an explicit stack (depth-first) or a `deque` (breadth-first) does the same job with no limit, agreeing with `scipy.ndimage.label`.

**Next time (S18): binary search trees and kd-trees** - a structure where recursion is exactly right, and search costs scale with the height of the tree.